# Concaveman3D vs $\alpha$-shape

In [15]:
import csv
import sys
from math import sin

import alphashape
import numpy as np
import pyvista as pv
from IPython.display import IFrame
from numpy.typing import NDArray

# pv.global_theme.notebook = True
# pv.global_theme.trame.jupyter_extension_enabled = True
pv.global_theme.trame.server_proxy_enabled = False
pv.global_theme.background = "1f1f1f"
pv.set_jupyter_backend("client")


def load_normalize(path: str):
    with open(path, newline="") as csvfile:
        reader = csv.reader(csvfile, quoting=csv.QUOTE_NONNUMERIC)
        points = np.array(list(reader))
        return normalize(points)


def normalize(points) -> NDArray[np.number]:
    v = np.subtract(points, np.min(points, axis=0))
    return v / np.max(v.flatten())


def add_mesh(pl: pv.Plotter, mesh):
    pv_mesh = pv.wrap(mesh)

    # Explicitly compute normals if they are missing
    # This ensures 'normals' are available in the point_data
    pv_mesh.compute_normals(inplace=True, point_normals=True, cell_normals=False)
    normal_colors = (pv_mesh.point_data["Normals"] + 1.0) / 2.0

    pl.add_mesh(
        pv_mesh,
        scalars=normal_colors,
        rgb=True,
        lighting=False,  # This makes it "unlit" like MeshNormalMaterial
        # smooth_shading=True,
        # preference='point'  # Ensures smooth gradients across faces
    )


def add_points(pl: pv.Plotter, points: NDArray[np.number]):
    d = pv.PolyData(points)
    d["Z_dist"] = 1 - np.abs(points[:, 2] - 0.5)
    pl.add_mesh(
        d.glyph(geom=pv.Sphere(radius=0.007), scale=False, orient=False),  # pyright: ignore[reportArgumentType]
        scalars="Z_dist",
        clim=[0, 1],
        show_scalar_bar=False,
    )


def plot_mesh(mesh):
    pl = pv.Plotter()
    add_mesh(pl, mesh)
    pl.show()


In [9]:
IFrame(src="https://kayjay7.github.io/concaveman3d", width="100%", height=500)

In [10]:
pl = pv.Plotter(shape=(1, 3))
add_mesh(
    pl,
    alphashape.alphashape(
        load_normalize("../inst/js/static/datasets/points3d.csv"),
        6,
    ),
)
pl.subplot(0, 1)
add_mesh(
    pl,
    alphashape.alphashape(
        load_normalize("../inst/js/static/datasets/chair.csv"),
        3.9,
    ),
)
pl.subplot(0, 2)
add_mesh(
    pl,
    alphashape.alphashape(
        load_normalize("../inst/js/static/datasets/mug.csv"),
        9.549,
    ),
)
pl.show()

d:\Users\Nemo\OneDrive-shared\es\concaveman3d\slides\.venv\Lib\site-packages\alphashape\alphashape.py:79: UserWarning: Singular matrix. Likely caused by all points lying in an N-1 space.
  warnings.warn('Singular matrix. Likely caused by all points '


Widget(value='<iframe src="http://localhost:57342/index.html?ui=P_0x2b0af930550_2&reconnect=auto" class="pyvis…

## Statistic demo

In [37]:
def query(x, y, z):
    a = 2.44
    b = 4.12
    c = 13.7
    d = 0.5
    return x**2 + y**2 <= c * sin(abs(z / b) ** a) + d


n_points = 2000

np.random.seed(7825204)
points = np.random.uniform([-5, -5, -7], [5, 5, 7], size=(n_points, 3))
points = normalize(
    points[np.vectorize(query)(points[:, 0], points[:, 1], points[:, 2])]
)

pl = pv.Plotter(shape=(1, 4))
add_points(pl, points)
# pl.export_gltf("assets/hourglass.gltf")
pl.subplot(0, 1)
add_mesh(pl, alphashape.alphashape(points, sys.float_info.min))
pl.subplot(0, 2)
add_mesh(pl, alphashape.alphashape(points, 8))
pl.subplot(0, 3)
add_mesh(pl, alphashape.alphashape(points, 17.2))
pl.show()


Widget(value='<iframe src="http://localhost:57342/index.html?ui=P_0x2b0dbc434d0_28&reconnect=auto" class="pyvi…